# PawCity AI — Gemini Vertex AI Verification

**Model:** `gemini-3.5-flash-lite` · **Provider:** Google Cloud Vertex AI · **Path:** Deep Research

This notebook uses the school Google Cloud project and its attached billing account, not the OpenAI API. First run `gcloud auth application-default login` in Mac Terminal, and ensure the school project has the Vertex AI API and IAM permissions enabled. Google Cloud inference can incur charges; confirm whether course credits apply.


## 1. Configure the school Google Cloud project

Replace `YOUR_SCHOOL_GCP_PROJECT_ID` with the exact ID shown in the Google Cloud project selector. Do not enter the Google Cloud project display name, OpenAI project ID, or any API key.


In [ ]:
import os
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / "app.py").exists(), "Open this notebook from the PawCity project root."

PROJECT_ID = "YOUR_SCHOOL_GCP_PROJECT_ID"
assert PROJECT_ID != "YOUR_SCHOOL_GCP_PROJECT_ID", "Replace PROJECT_ID with your actual school GCP Project ID."
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["PAWCITY_MODEL"] = "gemini-3.5-flash-lite"
os.environ["PAWCITY_API_RPM"] = "7"
print("Google Cloud Project:", PROJECT_ID)
print("Model:", os.environ["PAWCITY_MODEL"])


## 2. Verify the Gemini model using one real request

This step makes a small, billable Vertex AI request. A successful response confirms that the configured identity, Google Cloud project, and Gemini model are usable locally. It does not prove Cloud Run's service account already has access.


In [ ]:
from pawcity.research_agent import GeminiModel
model = GeminiModel()
print(model.request("Reply with exactly: Vertex AI connection OK"))


## 3. Run one real research question

The Lead, parallel Researchers, gap review, and Writer are run with Gemini. Their tool calls retain the Project 1 response shape and add an `agent` field.


In [ ]:
from pawcity.research_agent import PawCityResearch
agent = PawCityResearch(ROOT, llm=model, max_turns=2)
result = agent.run("Compare ZIP 10001 and ZIP 10002 using documented dog runs, license periods, 2025 barking reports and bite reports.")
print(result.answer)
print("Sub-agent tool calls:", [x["agent"] for x in result.tool_calls])
print("Research trace:", result.agent_trace)


## 4. Complete the required three comparisons

The project README requires three genuine paired results using the **same Gemini model** with and without tools or system instructions. The script checkpoints each result and updates the README once all three are complete. Legacy GPT checkpoints cannot be reused.


In [ ]:
import runpy
runpy.run_path(str(ROOT / "scripts" / "compare_samples.py"), run_name="__main__")
